**The following code has been implemented on Kaggle T4 GPU**

### Installation of STELLAR

In [1]:
!git clone -q https://github.com/microsoft/STELLAR.git
!ls -la /kaggle/working/STELLAR
%cd /kaggle/working/STELLAR

total 84
drwxr-xr-x  8 root root  4096 Sep 12 09:26 .
drwxr-xr-x  4 root root  4096 Sep 12 09:26 ..
drwxr-xr-x 10 root root  4096 Sep 12 09:26 configs
drwxr-xr-x  2 root root  4096 Sep 12 09:26 docs
drwxr-xr-x  2 root root  4096 Sep 12 09:26 examples
drwxr-xr-x  8 root root  4096 Sep 12 09:26 .git
-rw-r--r--  1 root root   136 Sep 12 09:26 .gitignore
drwxr-xr-x  2 root root  4096 Sep 12 09:26 images
-rw-r--r--  1 root root  1066 Sep 12 09:26 LICENSE
-rw-r--r--  1 root root  3409 Sep 12 09:26 load_stellar.py
-rw-r--r--  1 root root 20014 Sep 12 09:26 README.md
-rw-r--r--  1 root root    72 Sep 12 09:26 requirements-eval.txt
-rw-r--r--  1 root root   139 Sep 12 09:26 requirements-inference.txt
-rw-r--r--  1 root root   566 Sep 12 09:26 requirements.txt
-rw-r--r--  1 root root  1329 Sep 12 09:26 run.py
-rw-r--r--  1 root root   542 Sep 12 09:26 SECURITY.md
drwxr-xr-x  7 root root  4096 Sep 12 09:26 src
/kaggle/working/STELLAR


In [2]:
# Installing requirments.txt for STELLAR 
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.1 MB/s eta 0:00:00
ERROR: Ignored the following versions that require a different python version: 0.0.0 Requires-Python <3.11,>=3.8; 0.0.10 Requires-Python >=3.8,<3.9; 0.0.11 Requires-Python >=3.8,<3.9; 0.0.12 Requires-Python >=3.8,<3.9; 0.0.14 Requires-Python >=3.8,<3.9; 0.0.15 Requires-Python >=3.8,<3.9; 0.0.16 Requires-Python >=3.8,<3.9; 0.0.17 Requires-Python >=3.8,<3.9; 0.0.18 Requires-Python >=3.8,<3.9; 0.0.19 Requires-Python >=3.8,<3.9; 0.0.20 Requires-Python >=3.8,<3.9; 0.0.21 Requires-Python >=3.8,<3.9; 0.0.22 Requires-Python >=3.8,<3.9; 0.0.23 Requires-Python >=3.8,<3.9; 0.0.24 Requires-Python >=3.8,<3.9; 0.0.25 Requires-Python >=3.8,<3.9; 

In [3]:
# Installing safetensors for huggingface_hub for STELLAR working
!pip install -q huggingface_hub safetensors

In [6]:
# Loading baseline STELLAR
import json
import torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from src.models.stellar_model import STELLARModel # Loading the STELLAR-B16 model

repo = "microsoft/STELLAR" # STELLAR path
config_path = hf_hub_download(repo_id=repo,filename="config.json") # Configuration path

with open(config_path, "r") as f:
    config = json.load(f)
cfg = config["models"]["stellar-b16"] # Loading STELLAR-B16 model (more efficient)

# Loading model with configuration
model = STELLARModel(
    num_sparse_tokens=cfg["num_sparse_tokens"],
    num_decoder_layers=cfg["num_decoder_layers"],
    spatial_temp=cfg["spatial_temp"],
    vit_pretrained=cfg["backbone"],
    do_recon=False,
    do_clustering=False,
    vq_model=None,
)

# Setting up device for evaluation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()


You are using a model of type vit_mae to instantiate a model of type vit. This is not supported for all configurations of models and can yield errors.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: facebook/vit-mae-base
Key                                                                              | Status     | 
---------------------------------------------------------------------------------+------------+-
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.layernorm_after.weight           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.attention.value.bias   | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.attention.query.bias   | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.output.dense.weight              | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.intermediate.dense.weight        | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.output.dense.bias      | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.intermediate.dense.bias          | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.attention.key.weight

STELLARModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermediate_act_

In [7]:
"""
Sanity test on STELLAR-B16 to check semantic and localization matrix

For device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
"""
image = torch.rand(1, 3, 224, 224,device=device)
with torch.no_grad():
    out = model.encode(image)

for key, value in out.items():
    if torch.is_tensor(value):
        print(key, value.shape)
"""
sparse torch.Size([]) is semantic matrix
spatial torch.Size([]) is localization matrix
"""

sparse torch.Size([1, 16, 768])
spatial torch.Size([1, 196, 16])
concat torch.Size([1, 16, 964])
lowrank torch.Size([1, 196, 768])
dense torch.Size([1, 196, 768])
cls torch.Size([1, 1, 768])
peak_dist torch.Size([1, 16])


In Glas segmentation dataset, we'll be using a total of 85 images, out of which, 70 are for training and 15 are for testing. For testing purpose, we'll be using 60 TestA and 20 TestB images so far. 

Images in GlaS segmentation dataset are in .bmp (bitmap) format. In .bmp images, pixel values are explicitly stored and it generally lossless/uncompressed and can preserve exact pixel values. 

### GlaSSegmentationDataset

In [8]:

import os
import numpy as np
import torch

from PIL import Image
from torch.utils.data import Dataset
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode


class GlaSSegmentationDataset(Dataset):
    def __init__(self,root,split="train",resolution=224,crop_scale=0.5,augment=False,):
        self.root = root
        self.split = split
        self.resolution = resolution
        self.crop_scale = crop_scale
        self.augment = augment


        # Select images belonging to the official split
        if split == "train":
            image_files = [f for f in os.listdir(root)if f.startswith("train_")
                and f.endswith(".bmp") and not f.endswith("_anno.bmp")]
            
        elif split == "testA":
            image_files = [f for f in os.listdir(root) if f.startswith("testA_")
                and f.endswith(".bmp") and not f.endswith("_anno.bmp")]

        elif split == "testB":
            image_files = [f for f in os.listdir(root) if f.startswith("testB_")
                and f.endswith(".bmp") and not f.endswith("_anno.bmp")]

        else:
            raise ValueError( f"Unknown split: {split}. Expected train, testA or testB.")

        # Build image/mask pairs
        self.samples = []
        for image_file in sorted(image_files):
            mask_file = image_file.replace(".bmp","_anno.bmp")
            image_path = os.path.join(root,image_file)
            mask_path = os.path.join(root,mask_file)
            if not os.path.exists(mask_path):
                raise FileNotFoundError(f"Missing mask for {image_file} | {mask_path}")
            self.samples.append((image_path, mask_path))
        print(f"GlaS {split} | {len(self.samples)} image/mask pairs | augment={self.augment}")
        
    def __len__(self):
        return len(self.samples)


    """
    Instance mask to binary semantic mask
    
    We'll be converting instance mask to binary semantic mask
    In original GlaS dataset, 
    0      = background
    1..32  = individual gland instances

    In binary semantic masking, 
    0 = background
    1 = gland
    """
    def _convert_mask(self, mask):
        mask = np.array(mask)
        mask = (mask > 0).astype(np.uint8)
        return torch.from_numpy(mask).long()


    # Training augmentation
    def _train_transform(self, image, mask):
        width, height = TF.get_image_size(image)
        
        # Random crop scale
        scale = torch.empty(1).uniform_(self.crop_scale,1.0).item()

        crop_h = int(height * scale)
        crop_w = int(width * scale)

        crop_h = max(1, min(crop_h, height))
        crop_w = max(1, min(crop_w, width))

        # Random crop position
        if height == crop_h:
            top = 0
        else:
            top = torch.randint(0,height - crop_h + 1,(1,)).item()
            
        if width == crop_w:
            left = 0
        else:
            left = torch.randint(0,width - crop_w + 1,(1,)).item()

        # Image: bicubic
        image = TF.resized_crop(image,top,left,crop_h,crop_w,
            [self.resolution, self.resolution],
            interpolation=InterpolationMode.BICUBIC,)

        # Mask: nearest-neighbor
        mask = TF.resized_crop(mask.unsqueeze(0).float(),top,left,crop_h,crop_w,
            [self.resolution, self.resolution],
            interpolation=InterpolationMode.NEAREST,).squeeze(0).long()
     
        # Random horizontal flip
        if torch.rand(1).item() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)
        return image, mask
   
    # Deterministic validation/test transform
    def _eval_transform(self, image, mask):
        crop_res = int(self.resolution * 256 / 224)

        # Image: bicubic
        image = TF.resize(
            image,
            [crop_res, crop_res],
            interpolation=InterpolationMode.BICUBIC,
        )

        # Mask: nearest
        mask = TF.resize(
            mask.unsqueeze(0).float(),
            [crop_res, crop_res],
            interpolation=InterpolationMode.NEAREST,
        ).squeeze(0).long()

        # Center crop
        image = TF.center_crop(image,[self.resolution, self.resolution])
        mask = TF.center_crop(mask,[self.resolution, self.resolution])
        return image, mask

    # Get item
    def __getitem__(self, idx):
        image_path, mask_path = self.samples[idx]

        # Load image and mask
        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        mask = self._convert_mask(mask) # Instance to Semantic conversion

        # Apply transform
        if self.augment:
            image, mask = self._train_transform(image,mask) # For training
        else:
            image, mask = self._eval_transform(image,mask) # For testing/validation
        image = TF.to_tensor(image) # Image to Tensor transform
        return {
            "image": image.clamp(0, 1),
            "labels": mask,
            "filename": idx,
        }
        

In [9]:
# ROOT path of GlaS dataset
ROOT = "/kaggle/input/datasets/sani84/glasmiccai2015-gland-segmentation/Warwick_QU_Dataset"

# For training during training
glas_train_aug = GlaSSegmentationDataset(root=ROOT,split="train",resolution=224,augment=True,)
# For validation during training
glas_train_eval = GlaSSegmentationDataset(root=ROOT,split="train",resolution=224,augment=False,)

SEED = 42
generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(glas_train_aug),generator=generator).tolist()

train_indices = indices[:70]
val_indices = indices[70:]
print("Train indices:", len(train_indices))
print("Validation indices:", len(val_indices))

print("Train examples:", train_indices[:10])
print("Val examples:", val_indices[:10])

GlaS train | 85 image/mask pairs | augment=True
GlaS train | 85 image/mask pairs | augment=False
Train indices: 70
Validation indices: 15
Train examples: [47, 24, 37, 11, 13, 60, 17, 27, 71, 22]
Val examples: [23, 16, 75, 55, 39, 68, 41, 3, 62, 35]


### Train Loader, Test Loader

In [10]:
from torch.utils.data import Subset
from torch.utils.data import DataLoader

# Training subset
train_subset = Subset(glas_train_aug,train_indices)
# Validation subset
val_subset = Subset(glas_train_eval,val_indices)

# For testing purpose these testA_dataset, testB_dataset will be used
testA_dataset = GlaSSegmentationDataset(root=ROOT,split="testA",resolution=224,augment=False,)
testB_dataset = GlaSSegmentationDataset(root=ROOT,split="testB",resolution=224,augment=False,)

# Training loader
probe_train_loader = DataLoader(train_subset,batch_size=4,shuffle=True,num_workers=2,pin_memory=True,)
# Validation loader
probe_val_loader = DataLoader(val_subset,batch_size=4,shuffle=False,num_workers=2,pin_memory=True,)

# TestA loader
testA_loader = DataLoader(testA_dataset,batch_size=4,shuffle=False,num_workers=2,pin_memory=True,)
# TestB loader
testB_loader = DataLoader(testB_dataset,batch_size=4,shuffle=False,num_workers=2,pin_memory=True,)


GlaS testA | 60 image/mask pairs | augment=False
GlaS testB | 20 image/mask pairs | augment=False


### Segmentation Metrics

In [12]:
# segmentation metrics contain Dice, IoU, Accuracy
"""
Dice, IoU, Accuracy metrics is measured for image segmentation task
Four fundamental things is required for that, 
TP = True Positive = predicted foreground, actually foreground
FP = False Positive = predicted foreground, actually background
TN = True Negative = predicted background, actually background
FN = False Negative = predicted background, actually foreground

So, Dice = 2.TP/(2.TP + FP + FN) or Dice = 2|P intersection G| / |P| + |G|
where, P is the predicted segmentation and G is the ground truth.

IoU = TP/(TP + FP + FN) or IoU = |P intersection G| / |P union G|
mIoU = mean intersection over union (for every classes combined divided by total number of class)
Accuracy = TP+TN/(TP+TN+FP+FN)
"""


import numpy as np

def segmentation_metrics(pred, target, num_classes=2):
    pred = pred.detach().cpu().numpy().reshape(-1)
    target = target.detach().cpu().numpy().reshape(-1)
    metrics = {}
    ious = []

    for cls in range(num_classes):
        pred_cls = pred == cls
        target_cls = target == cls
        intersection = np.logical_and(pred_cls,target_cls).sum()
        union = np.logical_or(pred_cls,target_cls).sum()

        if union == 0:
            iou = np.nan
        else:
            iou = intersection / union
        ious.append(iou)

    # Background IoU and Gland IoU
    metrics["background_iou"] = ious[0]
    metrics["gland_iou"] = ious[1]
    valid_ious = [x for x in ious if not np.isnan(x)]
    metrics["miou"] = np.mean(valid_ious)

    # Gland Dice
    pred_gland = pred == 1
    target_gland = target == 1
    intersection = np.logical_and(pred_gland, target_gland).sum()
    pred_area = pred_gland.sum()
    target_area = target_gland.sum()
    denominator = pred_area + target_area

    if denominator == 0:
        dice = 1.0
    else:
        dice = (2.0 * intersection / denominator)

    metrics["gland_dice"] = dice
    metrics["pixel_accuracy"] = (pred == target).mean()
    return metrics

### Run Epoch Broilerplate Code

In [13]:
# Broiler plate running code for larger epochs
def run_epoch(model,loader,optimizer=None,criterion=None,device="cuda",):
    is_training = optimizer is not None
    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in loader:
        images = batch["image"].to(device)
        labels = batch["labels"].to(device)

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):
            outputs = model({"image": images})
            logits = outputs["predictions"]
            loss = criterion(logits,labels)
            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits,dim=1)
        all_preds.append(preds.detach().cpu())
        all_labels.append(labels.detach().cpu())

    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    metrics = segmentation_metrics(all_preds,all_labels)
    metrics["loss"] = (total_loss / len(loader.dataset))
    return metrics

### Dense Model 3 SEED

In [27]:
from src.models.downstream.segmentation import SegmentationProbing
dense_model = SegmentationProbing(
    model_backbone=model,
    is_baseline=False,
    feature_key="dense",
    feature_dim=768,
    num_classes=2,
    freeze_backbone=True,
    freeze_model=False,
    resize_output=(224, 224),
).to(device)

  # We'll be using Adam with weighted decay optimizer
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, dense_model.parameters()),
    lr=1e-3,weight_decay=1e-4,)
# We'll be using cross entropy loss
criterion = torch.nn.CrossEntropyLoss()

In [30]:
# Dense model running for seed 42, 123, 2026
import random

history = []

def train_dense_probe(seed, num_epochs=50):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Best checkpoint in memory
    best_val_dice = -1.0
    best_epoch = -1
    best_state = None

    # Training
    for epoch in range(num_epochs):
        train_metrics = run_epoch(dense_model,probe_train_loader,optimizer=optimizer,
            criterion=criterion,device=device,)
        
        val_metrics = run_epoch(dense_model,probe_val_loader,optimizer=None,
            criterion=criterion,device=device,)

        history.append({
            "epoch": epoch + 1,
            "train_dice": train_metrics["gland_dice"],
            "train_miou": train_metrics["miou"],
            "val_dice": val_metrics["gland_dice"],
            "val_miou": val_metrics["miou"],
        })

        print(
            f"[Seed {seed}] "
            f"Epoch {epoch+1:02d}/{num_epochs} | "
            f"Train Dice: {train_metrics['gland_dice']:.4f} | "
            f"Val Dice: {val_metrics['gland_dice']:.4f}"
        )

        if val_metrics["gland_dice"] > best_val_dice:
            best_val_dice = val_metrics["gland_dice"]
            best_epoch = epoch + 1
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in dense_model.state_dict().items()
            }


    """  
    # Restore best model
    dense_model.load_state_dict(best_state)
    dense_model.eval()

    # TestA
    testA_metrics = run_epoch(dense_model,testA_loader,optimizer=None,
        criterion=criterion,device=device,)

    # TestB
    testB_metrics = run_epoch(dense_model,testB_loader,optimizer=None,
        criterion=criterion,device=device,)
    """
    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_dice": best_val_dice,
        "history": history,
    }

In [34]:

dense_results = {}
for seed in [42, 123, 2026]:
    print("\n" + "=" * 70)
    print(f"STARTING DENSE PROBE — SEED {seed}")
    print("=" * 70)
    dense_results[seed] = train_dense_probe(seed=seed,num_epochs=50)


STARTING DENSE PROBE — SEED 42
[Seed 42] Epoch 01/50 | Train Dice: 0.8401 | Val Dice: 0.8482
[Seed 42] Epoch 02/50 | Train Dice: 0.8515 | Val Dice: 0.8380
[Seed 42] Epoch 03/50 | Train Dice: 0.8491 | Val Dice: 0.8381
[Seed 42] Epoch 04/50 | Train Dice: 0.8448 | Val Dice: 0.8389
[Seed 42] Epoch 05/50 | Train Dice: 0.8465 | Val Dice: 0.8459
[Seed 42] Epoch 06/50 | Train Dice: 0.8507 | Val Dice: 0.8430
[Seed 42] Epoch 07/50 | Train Dice: 0.8486 | Val Dice: 0.8440
[Seed 42] Epoch 08/50 | Train Dice: 0.8464 | Val Dice: 0.8495
[Seed 42] Epoch 09/50 | Train Dice: 0.8528 | Val Dice: 0.8485
[Seed 42] Epoch 10/50 | Train Dice: 0.8478 | Val Dice: 0.8503
[Seed 42] Epoch 11/50 | Train Dice: 0.8401 | Val Dice: 0.8420
[Seed 42] Epoch 12/50 | Train Dice: 0.8525 | Val Dice: 0.8450
[Seed 42] Epoch 13/50 | Train Dice: 0.8491 | Val Dice: 0.8449
[Seed 42] Epoch 14/50 | Train Dice: 0.8557 | Val Dice: 0.8425
[Seed 42] Epoch 15/50 | Train Dice: 0.8573 | Val Dice: 0.8477
[Seed 42] Epoch 16/50 | Train Dice: 0.

In [35]:
# storing values into csv file
import pandas as pd

history_df = pd.DataFrame(history)
csv_path = "/kaggle/working/stellar_dense_glas_probe_3_SEED_history.csv"
history_df.to_csv(csv_path, index=False)
display(history_df.head())

,epoch,train_dice,train_miou,val_dice,val_miou
0,1,0.793804,0.650547,0.808386,0.667846
1,2,0.808756,0.675887,0.812602,0.679120
2,3,0.815047,0.680883,0.819303,0.683343
3,4,0.824592,0.691928,0.818976,0.689919
4,1,0.826252,0.689748,0.834129,0.703021


In [36]:
# Summarizing results of 3 SEEDS for best validation dice
for seed, result in dense_results.items():
    print("\n" + "=" * 60)
    print(f"SEED {seed}")
    print("=" * 60)
    print(f"Best epoch: {result['best_epoch']}")
    print(f"Val Dice | {result['best_val_dice']:.4f}")


SEED 42
Best epoch: 41
Val Dice | 0.8542

SEED 123
Best epoch: 32
Val Dice | 0.8553

SEED 2026
Best epoch: 21
Val Dice | 0.8555


### Low Model 3 SEED

In [37]:
from src.models.downstream.segmentation import SegmentationProbing
lowrank_model = SegmentationProbing(
    model_backbone=model,
    is_baseline=False,
    feature_key="lowrank",
    feature_dim=768,
    num_classes=2,
    freeze_backbone=True,
    freeze_model=False,
    resize_output=(224, 224),
).to(device)


optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad,lowrank_model.parameters()),
    lr=1e-3,
    weight_decay=1e-4,)
criterion = torch.nn.CrossEntropyLoss()

In [39]:
import random

history=[]
# def train_spatial_probe(feature_key, seed, num_epochs=50):
def train_spatial_probe(seed, num_epochs=50):
    # Training randomness
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Best checkpoint in memory
    best_val_dice = -1.0
    best_epoch = -1
    best_state = None
    
    # Training
    for epoch in range(num_epochs):
        train_metrics = run_epoch(lowrank_model,probe_train_loader,optimizer=optimizer,
            criterion=criterion,
            device=device,)
        
        val_metrics = run_epoch(lowrank_model,probe_val_loader,optimizer=None,
            criterion=criterion,
            device=device,)

        history.append({
            "epoch": epoch + 1,
            "train_dice": train_metrics["gland_dice"],
            "train_miou": train_metrics["miou"],
            "val_dice": val_metrics["gland_dice"],
            "val_miou": val_metrics["miou"],
        })

        print(
            f"[Seed {seed}] "
            f"Epoch {epoch+1:02d}/{num_epochs} | "
            f"Train Dice: {train_metrics['gland_dice']:.4f} | "
            f"Val Dice: {val_metrics['gland_dice']:.4f}"
        )

        if val_metrics["gland_dice"] > best_val_dice:
            best_val_dice = val_metrics["gland_dice"]
            best_epoch = epoch + 1
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in lowrank_model.state_dict().items()
            }

    """ # Restore best model
        lowrank_model.load_state_dict(best_state)
        lowrank_model.eval()
    
    
        # TestA
        testA_metrics = run_epoch(
            lowrank_model,
            testA_loader,
            optimizer=None,
            criterion=criterion,
            device=device,
        )
    
    
        # TestB
        testB_metrics = run_epoch(
            lowrank_model,
            testB_loader,
            optimizer=None,
            criterion=criterion,
            device=device,
        )
    """
    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_dice": best_val_dice,
        "history": history,
    }
    

In [40]:
# Running for 3 SEEDS
lowrank_results = {}
for seed in [42, 123, 2026]:
    print("\n" + "=" * 70)
    print(f"STARTING LOWRANK PROBE — SEED {seed}")
    print("=" * 70)
    lowrank_results[seed] = train_spatial_probe(seed=seed,num_epochs=50,)
    


STARTING LOWRANK PROBE — SEED 42
[Seed 42] Epoch 01/50 | Train Dice: 0.5756 | Val Dice: 0.6853
[Seed 42] Epoch 02/50 | Train Dice: 0.5939 | Val Dice: 0.7033
[Seed 42] Epoch 03/50 | Train Dice: 0.6085 | Val Dice: 0.6850
[Seed 42] Epoch 04/50 | Train Dice: 0.6167 | Val Dice: 0.6671
[Seed 42] Epoch 05/50 | Train Dice: 0.6686 | Val Dice: 0.6691
[Seed 42] Epoch 06/50 | Train Dice: 0.6204 | Val Dice: 0.5891
[Seed 42] Epoch 07/50 | Train Dice: 0.6024 | Val Dice: 0.6232
[Seed 42] Epoch 08/50 | Train Dice: 0.6297 | Val Dice: 0.6143
[Seed 42] Epoch 09/50 | Train Dice: 0.6395 | Val Dice: 0.6414
[Seed 42] Epoch 10/50 | Train Dice: 0.6517 | Val Dice: 0.6461
[Seed 42] Epoch 11/50 | Train Dice: 0.6359 | Val Dice: 0.6138
[Seed 42] Epoch 12/50 | Train Dice: 0.6481 | Val Dice: 0.6585
[Seed 42] Epoch 13/50 | Train Dice: 0.6726 | Val Dice: 0.6495
[Seed 42] Epoch 14/50 | Train Dice: 0.6495 | Val Dice: 0.6173
[Seed 42] Epoch 15/50 | Train Dice: 0.6538 | Val Dice: 0.6566
[Seed 42] Epoch 16/50 | Train Dice: 

In [41]:
# storing values in .csv file
import pandas as pd
history_df = pd.DataFrame(history)
csv_path = "/kaggle/working/stellar_lowrank_glas_probe_3_SEED_history.csv"
history_df.to_csv(csv_path, index=False)

display(history_df.head())

,epoch,train_dice,train_miou,val_dice,val_miou
0,1,0.575565,0.376589,0.685261,0.316555
1,2,0.593883,0.407919,0.703326,0.457497
2,3,0.608490,0.418319,0.684990,0.385232
3,4,0.616734,0.426293,0.667081,0.426876
4,5,0.668614,0.460785,0.669109,0.408712


In [42]:
# Summarize results of lowrank
for seed, result in lowrank_results.items():
    print("\n" + "=" * 60)
    print(f"LOWRANK — SEED {seed}")
    print("=" * 60)
    print(f"Best epoch : {result['best_epoch']}")
    print(f"Val Dice   : {result['best_val_dice']:.4f}")


LOWRANK — SEED 42
Best epoch : 2
Val Dice   : 0.7033

LOWRANK — SEED 123
Best epoch : 17
Val Dice   : 0.7063

LOWRANK — SEED 2026
Best epoch : 32
Val Dice   : 0.7108


### Dense vs Low rank model Comparison

In [ ]:
# Comparison of Dense vs Low rank model

### Feature rank and Effective rank Broilerplate Code

In [43]:
# Feature rank
import torch
def feature_spectrum(x):
    x = x - x.mean(dim=0, keepdim=True)
    _, S, _ = torch.linalg.svd(x,full_matrices=False)
    return S

# Effective rank
def effective_rank(x):
    x = x.float()
    x = x - x.mean(dim=0, keepdim=True)
    _, S, _ = torch.linalg.svd(x,full_matrices=False)
    p = S / (S.sum() + 1e-12)
    entropy = -torch.sum(p * torch.log(p + 1e-12))
    return torch.exp(entropy).item()

In [45]:
# Feature rank and Effective rank of Dense and Low rank model
import torch

batch = next(iter(probe_val_loader))
images = batch["image"].to(device)
with torch.no_grad():
    outputs = model.encode({"image": images})

dense = outputs["dense"]
lowrank = outputs["lowrank"]

# Feature rank
x_dense = dense[0].float()
x_lowrank = lowrank[0].float()

S_dense = feature_spectrum(x_dense)
S_lowrank = feature_spectrum(x_lowrank)

print("Dense rank:", (S_dense > 1e-6).sum().item())
print("Lowrank rank:", (S_lowrank > 1e-6).sum().item())

print("Dense effective rank:",effective_rank(dense[0]))
print("Lowrank effective rank:",effective_rank(lowrank[0]))

Dense rank: 196
Lowrank rank: 24
Dense effective rank: 121.05934143066406
Lowrank effective rank: 8.088129997253418


### Rank Truncation Backbone

In [53]:
def truncate_spatial_rank(features, rank):
    """
    features: [B, N, C]
    N = spatial tokens
    C = feature dimension
    """
    B, N, C = features.shape
    output = []
    
    for b in range(B):
        X = features[b].float()
        
        # Center across spatial locations
        X_mean = X.mean(dim=0, keepdim=True)
        X_centered = X - X_mean

        # SVD
        U, S, Vh = torch.linalg.svd(X_centered,full_matrices=False)
        r = min(rank, S.shape[0])
        X_r = (U[:, :r] @ torch.diag(S[:r]) @ Vh[:r, :])

        # Restore mean
        X_r = X_r + X_mean
        output.append(X_r)
        
    return torch.stack(output)

In [54]:
# Rank truncation backbone (we're not directly changing SegmentationProbing)
class RankTruncatedBackbone(torch.nn.Module):
    def __init__(self, backbone, rank):
        super().__init__()
        self.backbone = backbone
        self.rank = rank

    def encode(self, inputs):
        outputs = self.backbone.encode(inputs)
        dense = outputs["dense"]
        dense_rank = truncate_spatial_rank(dense,self.rank)
        outputs = dict(outputs)
        outputs["dense"] = dense_rank
        return outputs

### Rank-32 Model 3 SEED

In [55]:
import torch.nn as nn
rank32_backbone = RankTruncatedBackbone(model,rank=32)

rank32_model = SegmentationProbing(
    model_backbone=rank32_backbone,
    is_baseline=False,
    feature_key="dense",
    feature_dim=768,
    num_classes=2,
    freeze_backbone=True,
    freeze_model=False,
    resize_output=(224, 224),
).to(device)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad,rank32_model.parameters()),
    lr=1e-3,weight_decay=1e-4,)
criterion = nn.CrossEntropyLoss()


In [56]:
import random

history = []
def train_spatial_probe(seed, num_epochs=50):
    # Training randomness
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Best checkpoint in memory
    best_val_dice = -1.0
    best_epoch = -1
    best_state = None

    # Training
    for epoch in range(num_epochs):
        train_metrics = run_epoch(rank32_model,probe_train_loader,optimizer=optimizer,
            criterion=criterion,device=device,)

        val_metrics = run_epoch(rank32_model,probe_val_loader,optimizer=None,
            criterion=criterion,device=device,)
        
        history.append({
            "epoch": epoch + 1,
            "train_dice": train_metrics["gland_dice"],
            "train_miou": train_metrics["miou"],
            "val_dice": val_metrics["gland_dice"],
            "val_miou": val_metrics["miou"],
        })

        print(
            f"[Seed {seed}] "
            f"Epoch {epoch+1:02d}/{num_epochs} | "
            f"Train Dice: {train_metrics['gland_dice']:.4f} | "
            f"Val Dice: {val_metrics['gland_dice']:.4f}"
        )

        if val_metrics["gland_dice"] > best_val_dice:
            best_val_dice = val_metrics["gland_dice"]
            best_epoch = epoch + 1
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in rank32_model.state_dict().items()
            }

    
    """ # Restore best model
        rank32_model.load_state_dict(best_state)
        rank32_model.eval()
        
        # TestA
        testA_metrics = run_epoch(
            rank32_model,
            testA_loader,
            optimizer=None,
            criterion=criterion,
            device=device,
        )

        # TestB
        testB_metrics = run_epoch(
            rank32_model,
            testB_loader,
            optimizer=None,
            criterion=criterion,
            device=device,
        )
    """
    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_dice": best_val_dice,
        "history": history,
    }

In [57]:
rank32_results = {}

for seed in [42, 123, 2026]:
    print("\n" + "=" * 70)
    print(f"STARTING RANK-32 PROBE — SEED {seed}")
    print("=" * 70)
    rank32_results[seed] = train_spatial_probe(seed=seed,num_epochs=50,)


STARTING LOWRANK PROBE — SEED 42
[Seed 42] Epoch 01/50 | Train Dice: 0.7267 | Val Dice: 0.7822
[Seed 42] Epoch 02/50 | Train Dice: 0.7974 | Val Dice: 0.7931
[Seed 42] Epoch 03/50 | Train Dice: 0.8011 | Val Dice: 0.8107
[Seed 42] Epoch 04/50 | Train Dice: 0.8169 | Val Dice: 0.8111
[Seed 42] Epoch 05/50 | Train Dice: 0.8168 | Val Dice: 0.8254
[Seed 42] Epoch 06/50 | Train Dice: 0.8306 | Val Dice: 0.8310
[Seed 42] Epoch 07/50 | Train Dice: 0.8274 | Val Dice: 0.8287
[Seed 42] Epoch 08/50 | Train Dice: 0.8251 | Val Dice: 0.8337
[Seed 42] Epoch 09/50 | Train Dice: 0.8318 | Val Dice: 0.8342
[Seed 42] Epoch 10/50 | Train Dice: 0.8314 | Val Dice: 0.8331
[Seed 42] Epoch 11/50 | Train Dice: 0.8275 | Val Dice: 0.8286
[Seed 42] Epoch 12/50 | Train Dice: 0.8370 | Val Dice: 0.8275
[Seed 42] Epoch 13/50 | Train Dice: 0.8333 | Val Dice: 0.8311
[Seed 42] Epoch 14/50 | Train Dice: 0.8417 | Val Dice: 0.8293
[Seed 42] Epoch 15/50 | Train Dice: 0.8467 | Val Dice: 0.8353
[Seed 42] Epoch 16/50 | Train Dice: 

In [58]:
# storing values into .csv file
import pandas as pd

history_df = pd.DataFrame(history)
csv_path = "/kaggle/working/stellar_rank32_glas_probe_3_SEED_history.csv"
history_df.to_csv(csv_path, index=False)

display(history_df.head())

,epoch,train_dice,train_miou,val_dice,val_miou
0,1,0.726652,0.549768,0.782203,0.614699
1,2,0.797403,0.656159,0.793134,0.649338
2,3,0.801064,0.658561,0.810676,0.674419
3,4,0.816922,0.680412,0.811122,0.681192
4,5,0.816797,0.680826,0.825370,0.688788


In [59]:
# Summarize results of lowrank
for seed, result in rank32_results.items():
    print("\n" + "=" * 60)
    print(f"RANK-32 — SEED {seed}")
    print("=" * 60)
    print(f"Best epoch : {result['best_epoch']}")
    print(f"Val Dice   : {result['best_val_dice']:.4f}")


RANK-32 — SEED 42
Best epoch : 41
Val Dice   : 0.8507

RANK-32 — SEED 123
Best epoch : 6
Val Dice   : 0.8537

RANK-32 — SEED 2026
Best epoch : 21
Val Dice   : 0.8527


In [62]:
# Feature rank and Effective rank for Rank-32 model
import torch

batch = next(iter(probe_val_loader))
images = batch["image"].to(device)
with torch.no_grad():
    outputs = rank32_backbone.encode({"image": images})

rank32_features = outputs["dense"]
X = rank32_features[0]

# Center exactly as done during truncation
X_centered = X - X.mean(dim=0, keepdim=True)
S = torch.linalg.svdvals(X_centered)
numerical_rank = (S > 1e-5).sum().item()
print("Numerical rank:", numerical_rank)
print("Rank-32 effective rank:",effective_rank(rank32_features[0]))


Numerical rank: 32
Rank-32 effective rank: 26.149513244628906


### Rank-64 Model 3 SEED

In [78]:
import torch.nn as nn
rank64_backbone = RankTruncatedBackbone(model,rank=64)

rank64_model = SegmentationProbing(
    model_backbone=rank64_backbone,
    is_baseline=False,
    feature_key="dense",
    feature_dim=768,
    num_classes=2,
    freeze_backbone=True,
    freeze_model=False,
    resize_output=(224, 224),
).to(device)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad,rank64_model.parameters()),
    lr=1e-3,weight_decay=1e-4,)
criterion = nn.CrossEntropyLoss()


In [79]:
import random

history = []
def train_spatial_probe(seed, num_epochs=50):
    # Training randomness
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Best checkpoint in memory
    best_val_dice = -1.0
    best_epoch = -1
    best_state = None

    # Training
    for epoch in range(num_epochs):
        train_metrics = run_epoch(rank64_model,probe_train_loader,optimizer=optimizer,
            criterion=criterion,device=device,)

        val_metrics = run_epoch(rank64_model,probe_val_loader,optimizer=None,
            criterion=criterion,device=device,)
        
        history.append({
            "epoch": epoch + 1,
            "train_dice": train_metrics["gland_dice"],
            "train_miou": train_metrics["miou"],
            "val_dice": val_metrics["gland_dice"],
            "val_miou": val_metrics["miou"],
        })

        print(
            f"[Seed {seed}] "
            f"Epoch {epoch+1:02d}/{num_epochs} | "
            f"Train Dice: {train_metrics['gland_dice']:.4f} | "
            f"Val Dice: {val_metrics['gland_dice']:.4f}"
        )

        if val_metrics["gland_dice"] > best_val_dice:
            best_val_dice = val_metrics["gland_dice"]
            best_epoch = epoch + 1
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in rank64_model.state_dict().items()
            }

    
    """ # Restore best model
        rank32_model.load_state_dict(best_state)
        rank32_model.eval()
        
        # TestA
        testA_metrics = run_epoch(
            rank32_model,
            testA_loader,
            optimizer=None,
            criterion=criterion,
            device=device,
        )

        # TestB
        testB_metrics = run_epoch(
            rank32_model,
            testB_loader,
            optimizer=None,
            criterion=criterion,
            device=device,
        )
    """
    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_dice": best_val_dice,
        "history": history,
    }

In [80]:
rank64_results = {}

for seed in [42, 123, 2026]:
    print("\n" + "=" * 70)
    print(f"STARTING RANK-64 PROBE — SEED {seed}")
    print("=" * 70)
    rank64_results[seed] = train_spatial_probe(seed=seed,num_epochs=50,)


STARTING RANK-64 PROBE — SEED 42
[Seed 42] Epoch 01/50 | Train Dice: 0.7008 | Val Dice: 0.7862
[Seed 42] Epoch 02/50 | Train Dice: 0.7899 | Val Dice: 0.8040
[Seed 42] Epoch 03/50 | Train Dice: 0.7988 | Val Dice: 0.8109
[Seed 42] Epoch 04/50 | Train Dice: 0.8156 | Val Dice: 0.8141
[Seed 42] Epoch 05/50 | Train Dice: 0.8189 | Val Dice: 0.8317
[Seed 42] Epoch 06/50 | Train Dice: 0.8310 | Val Dice: 0.8363
[Seed 42] Epoch 07/50 | Train Dice: 0.8258 | Val Dice: 0.8352
[Seed 42] Epoch 08/50 | Train Dice: 0.8262 | Val Dice: 0.8383
[Seed 42] Epoch 09/50 | Train Dice: 0.8366 | Val Dice: 0.8360
[Seed 42] Epoch 10/50 | Train Dice: 0.8328 | Val Dice: 0.8358
[Seed 42] Epoch 11/50 | Train Dice: 0.8299 | Val Dice: 0.8316
[Seed 42] Epoch 12/50 | Train Dice: 0.8378 | Val Dice: 0.8320
[Seed 42] Epoch 13/50 | Train Dice: 0.8374 | Val Dice: 0.8351
[Seed 42] Epoch 14/50 | Train Dice: 0.8444 | Val Dice: 0.8339
[Seed 42] Epoch 15/50 | Train Dice: 0.8485 | Val Dice: 0.8412
[Seed 42] Epoch 16/50 | Train Dice: 

In [81]:
# storing values into .csv file
import pandas as pd

history_df = pd.DataFrame(history)
csv_path = "/kaggle/working/stellar_rank64_glas_probe_3_SEED_history.csv"
history_df.to_csv(csv_path, index=False)

display(history_df.head())

,epoch,train_dice,train_miou,val_dice,val_miou
0,1,0.700831,0.526833,0.786231,0.630493
1,2,0.789943,0.649002,0.804036,0.662026
2,3,0.798801,0.656740,0.810905,0.672013
3,4,0.815564,0.679419,0.814112,0.683732
4,5,0.818853,0.685438,0.831655,0.696935


In [82]:
# Summarize results of lowrank
for seed, result in rank64_results.items():
    print("\n" + "=" * 60)
    print(f"RANK-64 — SEED {seed}")
    print("=" * 60)
    print(f"Best epoch : {result['best_epoch']}")
    print(f"Val Dice   : {result['best_val_dice']:.4f}")


RANK-64 — SEED 42
Best epoch : 41
Val Dice   : 0.8561

RANK-64 — SEED 123
Best epoch : 32
Val Dice   : 0.8552

RANK-64 — SEED 2026
Best epoch : 21
Val Dice   : 0.8558


In [83]:
# Feature rank and Effective rank for Rank-32 model
import torch

batch = next(iter(probe_val_loader))
images = batch["image"].to(device)
with torch.no_grad():
    outputs = rank64_backbone.encode({"image": images})

rank64_features = outputs["dense"]
X = rank64_features[0]

# Center exactly as done during truncation
X_centered = X - X.mean(dim=0, keepdim=True)
S = torch.linalg.svdvals(X_centered)
numerical_rank = (S > 1e-5).sum().item()
print("Numerical rank:", numerical_rank)
print("Rank-64 effective rank:",effective_rank(rank64_features[0]))


Numerical rank: 64
Rank-64 effective rank: 49.597938537597656


### Rank-16 for 3 SEED

In [84]:
import torch.nn as nn
rank16_backbone = RankTruncatedBackbone(model,rank=16)

rank16_model = SegmentationProbing(
    model_backbone=rank16_backbone,
    is_baseline=False,
    feature_key="dense",
    feature_dim=768,
    num_classes=2,
    freeze_backbone=True,
    freeze_model=False,
    resize_output=(224, 224),
).to(device)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad,rank16_model.parameters()),
    lr=1e-3,weight_decay=1e-4,)
criterion = nn.CrossEntropyLoss()


In [85]:
import random

history = []
def train_spatial_probe(seed, num_epochs=50):
    # Training randomness
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Best checkpoint in memory
    best_val_dice = -1.0
    best_epoch = -1
    best_state = None

    # Training
    for epoch in range(num_epochs):
        train_metrics = run_epoch(rank16_model,probe_train_loader,optimizer=optimizer,
            criterion=criterion,device=device,)

        val_metrics = run_epoch(rank16_model,probe_val_loader,optimizer=None,
            criterion=criterion,device=device,)
        
        history.append({
            "epoch": epoch + 1,
            "train_dice": train_metrics["gland_dice"],
            "train_miou": train_metrics["miou"],
            "val_dice": val_metrics["gland_dice"],
            "val_miou": val_metrics["miou"],
        })

        print(
            f"[Seed {seed}] "
            f"Epoch {epoch+1:02d}/{num_epochs} | "
            f"Train Dice: {train_metrics['gland_dice']:.4f} | "
            f"Val Dice: {val_metrics['gland_dice']:.4f}"
        )

        if val_metrics["gland_dice"] > best_val_dice:
            best_val_dice = val_metrics["gland_dice"]
            best_epoch = epoch + 1
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in rank16_model.state_dict().items()
            }

    
    """ # Restore best model
        rank32_model.load_state_dict(best_state)
        rank32_model.eval()
        
        # TestA
        testA_metrics = run_epoch(
            rank32_model,
            testA_loader,
            optimizer=None,
            criterion=criterion,
            device=device,
        )

        # TestB
        testB_metrics = run_epoch(
            rank32_model,
            testB_loader,
            optimizer=None,
            criterion=criterion,
            device=device,
        )
    """
    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_dice": best_val_dice,
        "history": history,
    }

In [86]:
rank16_results = {}

for seed in [42, 123, 2026]:
    print("\n" + "=" * 70)
    print(f"STARTING RANK-16 PROBE — SEED {seed}")
    print("=" * 70)
    rank16_results[seed] = train_spatial_probe(
        feature_key="lowrank",seed=seed,num_epochs=50,)


STARTING RANK-16 PROBE — SEED 42
[Seed 42] Epoch 01/50 | Train Dice: 0.7064 | Val Dice: 0.7837
[Seed 42] Epoch 02/50 | Train Dice: 0.7847 | Val Dice: 0.8015
[Seed 42] Epoch 03/50 | Train Dice: 0.7957 | Val Dice: 0.8088
[Seed 42] Epoch 04/50 | Train Dice: 0.8043 | Val Dice: 0.8102
[Seed 42] Epoch 05/50 | Train Dice: 0.8040 | Val Dice: 0.8266
[Seed 42] Epoch 06/50 | Train Dice: 0.8231 | Val Dice: 0.8234
[Seed 42] Epoch 07/50 | Train Dice: 0.8157 | Val Dice: 0.8270
[Seed 42] Epoch 08/50 | Train Dice: 0.8147 | Val Dice: 0.8318
[Seed 42] Epoch 09/50 | Train Dice: 0.8234 | Val Dice: 0.8307
[Seed 42] Epoch 10/50 | Train Dice: 0.8241 | Val Dice: 0.8252
[Seed 42] Epoch 11/50 | Train Dice: 0.8170 | Val Dice: 0.8158
[Seed 42] Epoch 12/50 | Train Dice: 0.8286 | Val Dice: 0.8178
[Seed 42] Epoch 13/50 | Train Dice: 0.8261 | Val Dice: 0.8206
[Seed 42] Epoch 14/50 | Train Dice: 0.8297 | Val Dice: 0.8155
[Seed 42] Epoch 15/50 | Train Dice: 0.8356 | Val Dice: 0.8305
[Seed 42] Epoch 16/50 | Train Dice: 

In [87]:
# storing values into .csv file
import pandas as pd

history_df = pd.DataFrame(history)
csv_path = "/kaggle/working/stellar_rank16_glas_probe_3_SEED_history.csv"
history_df.to_csv(csv_path, index=False)

display(history_df.head())

,epoch,train_dice,train_miou,val_dice,val_miou
0,1,0.706395,0.534474,0.783676,0.625083
1,2,0.784676,0.641879,0.801478,0.655178
2,3,0.795725,0.650345,0.808759,0.667786
3,4,0.804342,0.661470,0.810246,0.677146
4,5,0.804013,0.663216,0.826579,0.687656


In [88]:
# Summarize results of lowrank
for seed, result in rank16_results.items():
    print("\n" + "=" * 60)
    print(f"RANK-16 — SEED {seed}")
    print("=" * 60)
    print(f"Best epoch : {result['best_epoch']}")
    print(f"Val Dice   : {result['best_val_dice']:.4f}")


RANK-16 — SEED 42
Best epoch : 50
Val Dice   : 0.8389

RANK-16 — SEED 123
Best epoch : 5
Val Dice   : 0.8416

RANK-16 — SEED 2026
Best epoch : 21
Val Dice   : 0.8415


In [89]:
# Feature rank and Effective rank for Rank-32 model
import torch

batch = next(iter(probe_val_loader))
images = batch["image"].to(device)
with torch.no_grad():
    outputs = rank16_backbone.encode({"image": images})

rank16_features = outputs["dense"]
X = rank16_features[0]

# Center exactly as done during truncation
X_centered = X - X.mean(dim=0, keepdim=True)
S = torch.linalg.svdvals(X_centered)
numerical_rank = (S > 1e-5).sum().item()
print("Numerical rank:", numerical_rank)
print("Rank-16 effective rank:",effective_rank(rank16_features[0]))


Numerical rank: 16
Rank-16 effective rank: 13.63166332244873


### Rank-15 for 3 SEED

In [90]:
import torch.nn as nn
rank15_backbone = RankTruncatedBackbone(model,rank=15)

rank15_model = SegmentationProbing(
    model_backbone=rank15_backbone,
    is_baseline=False,
    feature_key="dense",
    feature_dim=768,
    num_classes=2,
    freeze_backbone=True,
    freeze_model=False,
    resize_output=(224, 224),
).to(device)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad,rank15_model.parameters()),
    lr=1e-3,weight_decay=1e-4,)
criterion = nn.CrossEntropyLoss()


In [91]:
import random

history = []
def train_spatial_probe(seed, num_epochs=50):
    # Training randomness
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Best checkpoint in memory
    best_val_dice = -1.0
    best_epoch = -1
    best_state = None

    # Training
    for epoch in range(num_epochs):
        train_metrics = run_epoch(rank15_model,probe_train_loader,optimizer=optimizer,
            criterion=criterion,device=device,)

        val_metrics = run_epoch(rank15_model,probe_val_loader,optimizer=None,
            criterion=criterion,device=device,)
        
        history.append({
            "epoch": epoch + 1,
            "train_dice": train_metrics["gland_dice"],
            "train_miou": train_metrics["miou"],
            "val_dice": val_metrics["gland_dice"],
            "val_miou": val_metrics["miou"],
        })

        print(
            f"[Seed {seed}] "
            f"Epoch {epoch+1:02d}/{num_epochs} | "
            f"Train Dice: {train_metrics['gland_dice']:.4f} | "
            f"Val Dice: {val_metrics['gland_dice']:.4f}"
        )

        if val_metrics["gland_dice"] > best_val_dice:
            best_val_dice = val_metrics["gland_dice"]
            best_epoch = epoch + 1
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in rank15_model.state_dict().items()
            }

    
    """ # Restore best model
        rank32_model.load_state_dict(best_state)
        rank32_model.eval()
        
        # TestA
        testA_metrics = run_epoch(
            rank32_model,
            testA_loader,
            optimizer=None,
            criterion=criterion,
            device=device,
        )

        # TestB
        testB_metrics = run_epoch(
            rank32_model,
            testB_loader,
            optimizer=None,
            criterion=criterion,
            device=device,
        )
    """
    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_dice": best_val_dice,
        "history": history,
    }

In [92]:
rank15_results = {}

for seed in [42, 123, 2026]:
    print("\n" + "=" * 70)
    print(f"STARTING RANK-15 PROBE — SEED {seed}")
    print("=" * 70)
    rank15_results[seed] = train_spatial_probe(
        feature_key="lowrank",seed=seed,num_epochs=50,)


STARTING RANK-15 PROBE — SEED 42
[Seed 42] Epoch 01/50 | Train Dice: 0.7050 | Val Dice: 0.7865
[Seed 42] Epoch 02/50 | Train Dice: 0.7825 | Val Dice: 0.8052
[Seed 42] Epoch 03/50 | Train Dice: 0.7928 | Val Dice: 0.8102
[Seed 42] Epoch 04/50 | Train Dice: 0.8028 | Val Dice: 0.8123
[Seed 42] Epoch 05/50 | Train Dice: 0.8024 | Val Dice: 0.8297
[Seed 42] Epoch 06/50 | Train Dice: 0.8212 | Val Dice: 0.8253
[Seed 42] Epoch 07/50 | Train Dice: 0.8147 | Val Dice: 0.8301
[Seed 42] Epoch 08/50 | Train Dice: 0.8141 | Val Dice: 0.8334
[Seed 42] Epoch 09/50 | Train Dice: 0.8221 | Val Dice: 0.8323
[Seed 42] Epoch 10/50 | Train Dice: 0.8240 | Val Dice: 0.8260
[Seed 42] Epoch 11/50 | Train Dice: 0.8159 | Val Dice: 0.8172
[Seed 42] Epoch 12/50 | Train Dice: 0.8270 | Val Dice: 0.8193
[Seed 42] Epoch 13/50 | Train Dice: 0.8253 | Val Dice: 0.8229
[Seed 42] Epoch 14/50 | Train Dice: 0.8283 | Val Dice: 0.8162
[Seed 42] Epoch 15/50 | Train Dice: 0.8341 | Val Dice: 0.8306
[Seed 42] Epoch 16/50 | Train Dice: 

In [93]:
# storing values into .csv file
import pandas as pd

history_df = pd.DataFrame(history)
csv_path = "/kaggle/working/stellar_rank15_glas_probe_3_SEED_history.csv"
history_df.to_csv(csv_path, index=False)

display(history_df.head())

,epoch,train_dice,train_miou,val_dice,val_miou
0,1,0.704986,0.532802,0.786468,0.628757
1,2,0.782537,0.638898,0.805216,0.660283
2,3,0.792794,0.646277,0.810223,0.669091
3,4,0.802837,0.659241,0.812264,0.679602
4,5,0.802405,0.661024,0.829678,0.691689


In [94]:
# Summarize results of lowrank
for seed, result in rank15_results.items():
    print("\n" + "=" * 60)
    print(f"RANK-15 — SEED {seed}")
    print("=" * 60)
    print(f"Best epoch : {result['best_epoch']}")
    print(f"Val Dice   : {result['best_val_dice']:.4f}")


RANK-15 — SEED 42
Best epoch : 50
Val Dice   : 0.8398

RANK-15 — SEED 123
Best epoch : 5
Val Dice   : 0.8425

RANK-15 — SEED 2026
Best epoch : 21
Val Dice   : 0.8413


In [95]:
# Feature rank and Effective rank for Rank-32 model
import torch

batch = next(iter(probe_val_loader))
images = batch["image"].to(device)
with torch.no_grad():
    outputs = rank15_backbone.encode({"image": images})

rank15_features = outputs["dense"]
X = rank15_features[0]

# Center exactly as done during truncation
X_centered = X - X.mean(dim=0, keepdim=True)
S = torch.linalg.svdvals(X_centered)
numerical_rank = (S > 1e-5).sum().item()
print("Numerical rank:", numerical_rank)
print("Rank-15 effective rank:",effective_rank(rank15_features[0]))


Numerical rank: 15
Rank-15 effective rank: 12.839035987854004


### Rank-8 model for 3 SEED

In [96]:
import torch.nn as nn
rank8_backbone = RankTruncatedBackbone(model,rank=8)

rank8_model = SegmentationProbing(
    model_backbone=rank8_backbone,
    is_baseline=False,
    feature_key="dense",
    feature_dim=768,
    num_classes=2,
    freeze_backbone=True,
    freeze_model=False,
    resize_output=(224, 224),
).to(device)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad,rank8_model.parameters()),
    lr=1e-3,weight_decay=1e-4,)
criterion = nn.CrossEntropyLoss()


In [99]:
import random

history = []
def train_spatial_probe(seed, num_epochs=50):
    # Training randomness
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Best checkpoint in memory
    best_val_dice = -1.0
    best_epoch = -1
    best_state = None

    # Training
    for epoch in range(num_epochs):
        train_metrics = run_epoch(rank8_model,probe_train_loader,optimizer=optimizer,
            criterion=criterion,device=device,)

        val_metrics = run_epoch(rank8_model,probe_val_loader,optimizer=None,
            criterion=criterion,device=device,)
        
        history.append({
            "epoch": epoch + 1,
            "train_dice": train_metrics["gland_dice"],
            "train_miou": train_metrics["miou"],
            "val_dice": val_metrics["gland_dice"],
            "val_miou": val_metrics["miou"],
        })

        print(
            f"[Seed {seed}] "
            f"Epoch {epoch+1:02d}/{num_epochs} | "
            f"Train Dice: {train_metrics['gland_dice']:.4f} | "
            f"Val Dice: {val_metrics['gland_dice']:.4f}"
        )

        if val_metrics["gland_dice"] > best_val_dice:
            best_val_dice = val_metrics["gland_dice"]
            best_epoch = epoch + 1
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in rank8_model.state_dict().items()
            }

    
    """ # Restore best model
        rank32_model.load_state_dict(best_state)
        rank32_model.eval()
        
        # TestA
        testA_metrics = run_epoch(
            rank32_model,
            testA_loader,
            optimizer=None,
            criterion=criterion,
            device=device,
        )

        # TestB
        testB_metrics = run_epoch(
            rank32_model,
            testB_loader,
            optimizer=None,
            criterion=criterion,
            device=device,
        )
    """
    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_dice": best_val_dice,
        "history": history,
    }

In [101]:
rank8_results = {}

for seed in [42, 123, 2026]:
    print("\n" + "=" * 70)
    print(f"STARTING RANK-8 PROBE — SEED {seed}")
    print("=" * 70)
    rank8_results[seed] = train_spatial_probe(seed=seed,num_epochs=50,)


STARTING RANK-8 PROBE — SEED 42
[Seed 42] Epoch 01/50 | Train Dice: 0.7776 | Val Dice: 0.7892
[Seed 42] Epoch 02/50 | Train Dice: 0.7781 | Val Dice: 0.7980
[Seed 42] Epoch 03/50 | Train Dice: 0.7930 | Val Dice: 0.7921
[Seed 42] Epoch 04/50 | Train Dice: 0.7894 | Val Dice: 0.7952
[Seed 42] Epoch 05/50 | Train Dice: 0.7919 | Val Dice: 0.8067
[Seed 42] Epoch 06/50 | Train Dice: 0.8016 | Val Dice: 0.8069
[Seed 42] Epoch 07/50 | Train Dice: 0.7973 | Val Dice: 0.8070
[Seed 42] Epoch 08/50 | Train Dice: 0.7991 | Val Dice: 0.8173
[Seed 42] Epoch 09/50 | Train Dice: 0.8015 | Val Dice: 0.8115
[Seed 42] Epoch 10/50 | Train Dice: 0.8065 | Val Dice: 0.8091
[Seed 42] Epoch 11/50 | Train Dice: 0.8016 | Val Dice: 0.8069
[Seed 42] Epoch 12/50 | Train Dice: 0.8114 | Val Dice: 0.8088
[Seed 42] Epoch 13/50 | Train Dice: 0.8050 | Val Dice: 0.8059
[Seed 42] Epoch 14/50 | Train Dice: 0.8112 | Val Dice: 0.7926
[Seed 42] Epoch 15/50 | Train Dice: 0.8147 | Val Dice: 0.8115
[Seed 42] Epoch 16/50 | Train Dice: 0

In [102]:
# storing values into .csv file
import pandas as pd

history_df = pd.DataFrame(history)
csv_path = "/kaggle/working/stellar_rank8_glas_probe_3_SEED_history.csv"
history_df.to_csv(csv_path, index=False)

display(history_df.head())

,epoch,train_dice,train_miou,val_dice,val_miou
0,1,0.698239,0.521126,0.774365,0.608564
1,1,0.777578,0.625968,0.789211,0.637360
2,2,0.778144,0.631434,0.798050,0.647305
3,3,0.792974,0.644506,0.792148,0.634817
4,4,0.789365,0.638857,0.795159,0.649468


In [103]:
# Summarize results of lowrank
for seed, result in rank8_results.items():
    print("\n" + "=" * 60)
    print(f"RANK-15 — SEED {seed}")
    print("=" * 60)
    print(f"Best epoch : {result['best_epoch']}")
    print(f"Val Dice   : {result['best_val_dice']:.4f}")


RANK-15 — SEED 42
Best epoch : 50
Val Dice   : 0.8263

RANK-15 — SEED 123
Best epoch : 38
Val Dice   : 0.8307

RANK-15 — SEED 2026
Best epoch : 11
Val Dice   : 0.8261


In [104]:
# Feature rank and Effective rank for Rank-32 model
import torch

batch = next(iter(probe_val_loader))
images = batch["image"].to(device)
with torch.no_grad():
    outputs = rank8_backbone.encode({"image": images})

rank8_features = outputs["dense"]
X = rank8_features[0]

# Center exactly as done during truncation
X_centered = X - X.mean(dim=0, keepdim=True)
S = torch.linalg.svdvals(X_centered)
numerical_rank = (S > 1e-5).sum().item()
print("Numerical rank:", numerical_rank)
print("Rank-15 effective rank:",effective_rank(rank8_features[0]))


Numerical rank: 8
Rank-15 effective rank: 7.148799896240234


So, we find that, 
After taking average of all 3 SEED best val Dice
For Dense model, average best val Dice = 0.8555 
For Lowrank Mode, average best val Dice = 0.7068
For Rank-64 model, average best val Dice = 0.8557 (better than Dense model)
For Rank-32 model, average best val Dice = 0.8523
For Rank-16 model, average best val Dice = 0.8406
For Rank-15 model, average best val Dice = 0.8412
For Rank-8 model, average best val Dice = 0.8277

So, according to the model rank selection criteria based on val Dice, we can see that, for Rank-64, val Dice is more (0.02337%). So, we choose the Rank-64 model for TestA, TestB tests. 
Dense model rank = 196, Rank-64 model rank = 64, total compression = 67.34%

*Next step: taking Rank-64 model, test on TestA, TestB and compare performance against Dense and Lowrank model*

### Dense Model Testing

In [ ]:
from src.models.downstream.segmentation import SegmentationProbing

dense_model = SegmentationProbing(
    model_backbone=model,
    is_baseline=False,
    feature_key="dense",
    feature_dim=768,
    num_classes=2,
    freeze_backbone=True,
    freeze_model=False,
    resize_output=(224, 224),
).to(device)

  # We'll be using Adam with weighted decay optimizer
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, dense_model.parameters()),
    lr=1e-3,weight_decay=1e-4,)
# We'll be using cross entropy loss
criterion = torch.nn.CrossEntropyLoss()

In [105]:
# Dense model running for seed 42, 123, 2026
import random

history = []

def train_dense_probe(seed, num_epochs=50):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Best checkpoint in memory
    best_val_dice = -1.0
    best_epoch = -1
    best_state = None

    # Training
    for epoch in range(num_epochs):
        train_metrics = run_epoch(dense_model,probe_train_loader,optimizer=optimizer,
            criterion=criterion,device=device,)
        
        val_metrics = run_epoch(dense_model,probe_val_loader,optimizer=None,
            criterion=criterion,device=device,)

        history.append({
            "epoch": epoch + 1,
            "train_dice": train_metrics["gland_dice"],
            "train_miou": train_metrics["miou"],
            "val_dice": val_metrics["gland_dice"],
            "val_miou": val_metrics["miou"],
        })

        print(
            f"[Seed {seed}] "
            f"Epoch {epoch+1:02d}/{num_epochs} | "
            f"Train Dice: {train_metrics['gland_dice']:.4f} | "
            f"Val Dice: {val_metrics['gland_dice']:.4f}"
        )

        if val_metrics["gland_dice"] > best_val_dice:
            best_val_dice = val_metrics["gland_dice"]
            best_epoch = epoch + 1
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in dense_model.state_dict().items()
            }



    # Restore best model
    dense_model.load_state_dict(best_state)
    dense_model.eval()

    # TestA
    testA_metrics = run_epoch(dense_model,testA_loader,optimizer=None,
        criterion=criterion,device=device,)

    # TestB
    testB_metrics = run_epoch(dense_model,testB_loader,optimizer=None,
        criterion=criterion,device=device,)
    
    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_dice": best_val_dice,
        "testA": testA_metrics,
        "testB": testB_metrics,
        "history": history,
    }

In [106]:
dense_results = {}
for seed in [42, 123, 2026]:
    print("\n" + "=" * 70)
    print(f"STARTING DENSE PROBE — SEED {seed}")
    print("=" * 70)
    dense_results[seed] = train_dense_probe(seed=seed,num_epochs=50)


STARTING DENSE PROBE — SEED 42
[Seed 42] Epoch 01/50 | Train Dice: 0.8486 | Val Dice: 0.8512
[Seed 42] Epoch 02/50 | Train Dice: 0.8553 | Val Dice: 0.8491
[Seed 42] Epoch 03/50 | Train Dice: 0.8518 | Val Dice: 0.8507
[Seed 42] Epoch 04/50 | Train Dice: 0.8533 | Val Dice: 0.8485
[Seed 42] Epoch 05/50 | Train Dice: 0.8531 | Val Dice: 0.8514
[Seed 42] Epoch 06/50 | Train Dice: 0.8578 | Val Dice: 0.8449
[Seed 42] Epoch 07/50 | Train Dice: 0.8540 | Val Dice: 0.8483
[Seed 42] Epoch 08/50 | Train Dice: 0.8506 | Val Dice: 0.8473
[Seed 42] Epoch 09/50 | Train Dice: 0.8600 | Val Dice: 0.8496
[Seed 42] Epoch 10/50 | Train Dice: 0.8525 | Val Dice: 0.8518
[Seed 42] Epoch 11/50 | Train Dice: 0.8460 | Val Dice: 0.8492
[Seed 42] Epoch 12/50 | Train Dice: 0.8618 | Val Dice: 0.8503
[Seed 42] Epoch 13/50 | Train Dice: 0.8564 | Val Dice: 0.8493
[Seed 42] Epoch 14/50 | Train Dice: 0.8555 | Val Dice: 0.8466
[Seed 42] Epoch 15/50 | Train Dice: 0.8590 | Val Dice: 0.8491
[Seed 42] Epoch 16/50 | Train Dice: 0.

In [107]:
# Summary of Dense model result
def get_summary(results, split, metric):

    values = np.array([result[split][metric]for result in results.values()])
    return (values.mean(),values.std(ddof=1),)


for split in ["testA", "testB"]:
    print(f"\n===== {split.upper()} =====")
    for metric in ["gland_dice","gland_iou","miou",]:
        mean, std = get_summary( dense_results,split,metric,)
        print(f"{metric:15s} | {mean:.4f} ± {std:.4f}")


===== TESTA =====
gland_dice      | 0.8473 ± 0.0002
gland_iou       | 0.7351 ± 0.0002
miou            | 0.7296 ± 0.0008

===== TESTB =====
gland_dice      | 0.8483 ± 0.0001
gland_iou       | 0.7366 ± 0.0001
miou            | 0.6779 ± 0.0007


### Lowrank Model Testing

In [108]:
from src.models.downstream.segmentation import SegmentationProbing
lowrank_model = SegmentationProbing(
    model_backbone=model,
    is_baseline=False,
    feature_key="lowrank",
    feature_dim=768,
    num_classes=2,
    freeze_backbone=True,
    freeze_model=False,
    resize_output=(224, 224),
).to(device)


optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad,lowrank_model.parameters()),
    lr=1e-3,
    weight_decay=1e-4,)
criterion = torch.nn.CrossEntropyLoss()

In [109]:
import random

history=[]
# def train_spatial_probe(feature_key, seed, num_epochs=50):
def train_spatial_probe(seed, num_epochs=50):
    # Training randomness
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Best checkpoint in memory
    best_val_dice = -1.0
    best_epoch = -1
    best_state = None
    
    # Training
    for epoch in range(num_epochs):
        train_metrics = run_epoch(lowrank_model,probe_train_loader,optimizer=optimizer,
            criterion=criterion,
            device=device,)
        
        val_metrics = run_epoch(lowrank_model,probe_val_loader,optimizer=None,
            criterion=criterion,
            device=device,)

        history.append({
            "epoch": epoch + 1,
            "train_dice": train_metrics["gland_dice"],
            "train_miou": train_metrics["miou"],
            "val_dice": val_metrics["gland_dice"],
            "val_miou": val_metrics["miou"],
        })

        print(
            f"[Seed {seed}] "
            f"Epoch {epoch+1:02d}/{num_epochs} | "
            f"Train Dice: {train_metrics['gland_dice']:.4f} | "
            f"Val Dice: {val_metrics['gland_dice']:.4f}"
        )

        if val_metrics["gland_dice"] > best_val_dice:
            best_val_dice = val_metrics["gland_dice"]
            best_epoch = epoch + 1
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in lowrank_model.state_dict().items()
            }

    # Restore best model
    lowrank_model.load_state_dict(best_state)
    lowrank_model.eval()
    
    # TestA
    testA_metrics = run_epoch(lowrank_model,testA_loader,optimizer=None,
        criterion=criterion,device=device,)

    # TestB
    testB_metrics = run_epoch(lowrank_model,testB_loader,optimizer=None,
        criterion=criterion,device=device,)
    
    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_dice": best_val_dice,
        "testA": testA_metrics,
        "testB": testB_metrics,
        "history": history,
    }
    

In [110]:
# Running for 3 SEEDS
lowrank_results = {}
for seed in [42, 123, 2026]:
    print("\n" + "=" * 70)
    print(f"STARTING LOWRANK PROBE — SEED {seed}")
    print("=" * 70)
    lowrank_results[seed] = train_spatial_probe(seed=seed,num_epochs=50,)


STARTING LOWRANK PROBE — SEED 42
[Seed 42] Epoch 01/50 | Train Dice: 0.5567 | Val Dice: 0.6679
[Seed 42] Epoch 02/50 | Train Dice: 0.5959 | Val Dice: 0.6810
[Seed 42] Epoch 03/50 | Train Dice: 0.6009 | Val Dice: 0.6386
[Seed 42] Epoch 04/50 | Train Dice: 0.6184 | Val Dice: 0.6559
[Seed 42] Epoch 05/50 | Train Dice: 0.6523 | Val Dice: 0.6890
[Seed 42] Epoch 06/50 | Train Dice: 0.6187 | Val Dice: 0.6410
[Seed 42] Epoch 07/50 | Train Dice: 0.6206 | Val Dice: 0.6487
[Seed 42] Epoch 08/50 | Train Dice: 0.6048 | Val Dice: 0.6043
[Seed 42] Epoch 09/50 | Train Dice: 0.6517 | Val Dice: 0.6352
[Seed 42] Epoch 10/50 | Train Dice: 0.6639 | Val Dice: 0.6168
[Seed 42] Epoch 11/50 | Train Dice: 0.6562 | Val Dice: 0.5693
[Seed 42] Epoch 12/50 | Train Dice: 0.6572 | Val Dice: 0.6292
[Seed 42] Epoch 13/50 | Train Dice: 0.6718 | Val Dice: 0.6413
[Seed 42] Epoch 14/50 | Train Dice: 0.6566 | Val Dice: 0.5945
[Seed 42] Epoch 15/50 | Train Dice: 0.6645 | Val Dice: 0.6484
[Seed 42] Epoch 16/50 | Train Dice: 

In [111]:
# Summary of Lowrank model result
def get_summary(results, split, metric):

    values = np.array([result[split][metric]for result in results.values()])
    return (values.mean(),values.std(ddof=1),)


for split in ["testA", "testB"]:
    print(f"\n===== {split.upper()} =====")
    for metric in ["gland_dice","gland_iou","miou",]:
        mean, std = get_summary( lowrank_results,split,metric,)
        print(f"{metric:15s} | {mean:.4f} ± {std:.4f}")


===== TESTA =====
gland_dice      | 0.6154 ± 0.0184
gland_iou       | 0.4446 ± 0.0193
miou            | 0.3964 ± 0.0408

===== TESTB =====
gland_dice      | 0.7115 ± 0.0082
gland_iou       | 0.5523 ± 0.0098
miou            | 0.3987 ± 0.0453


### Rank-64 Model Testing

In [112]:
import torch.nn as nn
rank64_backbone = RankTruncatedBackbone(model,rank=64)

rank64_model = SegmentationProbing(
    model_backbone=rank64_backbone,
    is_baseline=False,
    feature_key="dense",
    feature_dim=768,
    num_classes=2,
    freeze_backbone=True,
    freeze_model=False,
    resize_output=(224, 224),
).to(device)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad,rank64_model.parameters()),
    lr=1e-3,weight_decay=1e-4,)
criterion = nn.CrossEntropyLoss()


In [113]:
import random

history = []
def train_spatial_probe(seed, num_epochs=50):
    # Training randomness
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Best checkpoint in memory
    best_val_dice = -1.0
    best_epoch = -1
    best_state = None

    # Training
    for epoch in range(num_epochs):
        train_metrics = run_epoch(rank64_model,probe_train_loader,optimizer=optimizer,
            criterion=criterion,device=device,)

        val_metrics = run_epoch(rank64_model,probe_val_loader,optimizer=None,
            criterion=criterion,device=device,)
        
        history.append({
            "epoch": epoch + 1,
            "train_dice": train_metrics["gland_dice"],
            "train_miou": train_metrics["miou"],
            "val_dice": val_metrics["gland_dice"],
            "val_miou": val_metrics["miou"],
        })

        print(
            f"[Seed {seed}] "
            f"Epoch {epoch+1:02d}/{num_epochs} | "
            f"Train Dice: {train_metrics['gland_dice']:.4f} | "
            f"Val Dice: {val_metrics['gland_dice']:.4f}"
        )

        if val_metrics["gland_dice"] > best_val_dice:
            best_val_dice = val_metrics["gland_dice"]
            best_epoch = epoch + 1
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in rank64_model.state_dict().items()
            }

    
    # Restore best model
    rank64_model.load_state_dict(best_state)
    rank64_model.eval()
    
    # TestA
    testA_metrics = run_epoch(rank64_model,testA_loader,optimizer=None,
        criterion=criterion,device=device,)

    # TestB
    testB_metrics = run_epoch(rank64_model,testB_loader,optimizer=None,
        criterion=criterion,device=device,)

    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_dice": best_val_dice,
        "testA": testA_metrics,
        "testB": testB_metrics,
        "history": history,
    }

In [114]:
# Running for 3 SEEDS
rank64_results = {}
for seed in [42, 123, 2026]:
    print("\n" + "=" * 70)
    print(f"STARTING RANK-64 PROBE — SEED {seed}")
    print("=" * 70)
    rank64_results[seed] = train_spatial_probe(seed=seed,num_epochs=50,)


STARTING RANK-64 PROBE — SEED 42
[Seed 42] Epoch 01/50 | Train Dice: 0.7187 | Val Dice: 0.7819
[Seed 42] Epoch 02/50 | Train Dice: 0.7926 | Val Dice: 0.7962
[Seed 42] Epoch 03/50 | Train Dice: 0.8026 | Val Dice: 0.8103
[Seed 42] Epoch 04/50 | Train Dice: 0.8194 | Val Dice: 0.8103
[Seed 42] Epoch 05/50 | Train Dice: 0.8198 | Val Dice: 0.8243
[Seed 42] Epoch 06/50 | Train Dice: 0.8347 | Val Dice: 0.8297
[Seed 42] Epoch 07/50 | Train Dice: 0.8270 | Val Dice: 0.8273
[Seed 42] Epoch 08/50 | Train Dice: 0.8288 | Val Dice: 0.8320
[Seed 42] Epoch 09/50 | Train Dice: 0.8386 | Val Dice: 0.8322
[Seed 42] Epoch 10/50 | Train Dice: 0.8338 | Val Dice: 0.8334
[Seed 42] Epoch 11/50 | Train Dice: 0.8300 | Val Dice: 0.8292
[Seed 42] Epoch 12/50 | Train Dice: 0.8382 | Val Dice: 0.8306
[Seed 42] Epoch 13/50 | Train Dice: 0.8386 | Val Dice: 0.8343
[Seed 42] Epoch 14/50 | Train Dice: 0.8456 | Val Dice: 0.8322
[Seed 42] Epoch 15/50 | Train Dice: 0.8500 | Val Dice: 0.8394
[Seed 42] Epoch 16/50 | Train Dice: 

In [115]:
# Summary of Lowrank model result
def get_summary(results, split, metric):

    values = np.array([result[split][metric]for result in results.values()])
    return (values.mean(),values.std(ddof=1),)


for split in ["testA", "testB"]:
    print(f"\n===== {split.upper()} =====")
    for metric in ["gland_dice","gland_iou","miou",]:
        mean, std = get_summary( rank64_results,split,metric,)
        print(f"{metric:15s} | {mean:.4f} ± {std:.4f}")


===== TESTA =====
gland_dice      | 0.8407 ± 0.0025
gland_iou       | 0.7252 ± 0.0037
miou            | 0.7244 ± 0.0022

===== TESTB =====
gland_dice      | 0.8400 ± 0.0017
gland_iou       | 0.7242 ± 0.0025
miou            | 0.6652 ± 0.0013


Despite reducing the representation rank from 196 to 64 (67.35% dimensionality reduction), the Rank-64 model retains performance close to the dense baseline across both held-out test sets. On TestA, gland Dice decreases by only 0.0066 (0.78%), while gland IoU and mIoU decrease by 0.0081 (1.10%) and 0.0052 (0.71%), respectively. On TestB, the corresponding decreases are 0.0083 (0.98%), 0.0124 (1.68%), and 0.0127 (1.87%). These results indicate that substantial representation compression can be achieved with only a minor degradation in segmentation performance. 